<a href="https://colab.research.google.com/github/arups330/ElitLab_MED_VQA/blob/main/Abdomen_open__CoT_finetune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# [Colab] Open-question fine-tuning **CoT** — all models, one after another, auto-push to Hugging Face

**Before running**
* Runtime → Change runtime type → **GPU** (T4).
* Upload your dataset **.zip** to `/content` via the Files panel (left sidebar). Cell 5 extracts it and finds `open_with_CoT.csv` automatically.
* 🔑 **Secrets** (left sidebar) → add `HF_TOKEN` = your Hugging Face **Write** token → enable *Notebook access*.
* Accept the license at https://huggingface.co/google/medgemma-4b-it.

This is the **CoT-conditioned** counterpart to the *WITHOUT CoT* open-question notebook. Two things change:

1. **Prompt** — the instruction now includes a *Reasoning Reference* (the CoT column from the CSV) that the model
   is told to verify against the image and use only if it agrees with the visual evidence.
2. **Data** — rows must have a non-empty answer **and** a non-empty CoT; rows with a missing CoT are dropped
   (there's nothing to condition on). The assistant target is still the ground-truth free-text answer (1–4 words).

Everything else (per-model steps, auto-skip already-pushed models, GPU cleanup between models) is identical.
Adapters are pushed as `{HF_USERNAME}/{DATASET_TAG}_open_CoT_{model}_lora` so they never collide with the no-CoT runs.

Cells 7–14 are each one step of the original notebook (load → LoRA → before check → trainer → train → after check → save → push → free GPU). Cell 15 runs them for every model in `REPO_IDS`, pushing each to the Hub before the next. Models already on the Hub are skipped — if the summary shows an OOM, **Runtime → Restart session** and run again.


In [18]:
!pip install -q "unsloth==2026.9.2"


In [19]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # before torch is imported
os.environ["UNSLOTH_RETURN_LOGITS"]   = "1"   # bypass Unsloth fused CE loss (Llama torch.compile crash)
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"   # disable torch.compile patches (same maths, a bit slower)

from unsloth import FastVisionModel # FastLanguageModel for LLMs
import torch, gc, shutil

REPO_IDS = {
    "MedGemma-4B": "google/medgemma-4b-it",
    "Qwen2.5-VL-7B": "Qwen/Qwen2.5-VL-7B-Instruct",
    "Qwen3-VL-8B": "Qwen/Qwen3-VL-8B-Instruct",
    "Lingshu-7B": "lingshu-medical-mllm/Lingshu-7B",
    "Llama-11B": "unsloth/Llama-3.2-11B-Vision-Instruct",

    # "Gemma4-E4B": "google/gemma-4-E4B-it",   # not trainable on T4 -> needs Colab L4/A100
}

HF_USERNAME = "Arup330"
DATASET_TAG = "Abdomen"


In [20]:
# HF token from Colab Secrets (🔑 icon in left sidebar -> add HF_TOKEN, enable notebook access)
from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN


In [21]:
# ---------------------------------------------------------------------------
# 3. Upload zip -> extract -> find open_with_CoT.csv -> keep rows with a valid answer AND a CoT
# ---------------------------------------------------------------------------
import os
import glob
import zipfile
import pandas as pd
from PIL import Image

EXTRACT_DIR = "/content/data"

# Use a zip already uploaded to /content (Files panel), otherwise open the upload dialog
zips = glob.glob("/content/*.zip")
if not zips:
    from google.colab import files
    uploaded = files.upload()
    zips = [f"/content/{n}" for n in uploaded.keys()]
ZIP_PATH = zips[0]
print("Using zip:", ZIP_PATH)

os.makedirs(EXTRACT_DIR, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall(EXTRACT_DIR)
print("Extracted to", EXTRACT_DIR)

# Locate open_with_CoT.csv anywhere inside the extracted folder (case-insensitive)
csv_paths = [p for p in glob.glob(os.path.join(EXTRACT_DIR, "**", "*.csv"), recursive=True)
             if os.path.basename(p).lower() == "open_with_cot.csv"]
assert csv_paths, "open_with_CoT.csv not found inside the zip"
print(f"Found {len(csv_paths)} CSVs:")
for p in csv_paths:
    print(" ", p)

frames = []
for csv_path in csv_paths:
    split_dir = os.path.dirname(csv_path)
    df = pd.read_csv(csv_path)
    df["split_dir"] = split_dir
    frames.append(df)

open_df = pd.concat(frames, ignore_index=True)
print("Columns:", list(open_df.columns))

# --- Find the CoT column (name varies between exports: cot / CoT / reasoning / chain_of_thought / ...) ---
def find_cot_column(columns):
    for c in columns:
        if c.lower() in ("cot", "chain_of_thought", "reasoning", "rationale", "explanation"):
            return c
    for c in columns:                       # looser match, e.g. "CoT_reasoning", "gpt_cot"
        if "cot" in c.lower() or "reason" in c.lower():
            return c
    raise KeyError(f"No CoT column found in {list(columns)} -- set COT_COL manually.")

COT_COL = find_cot_column(open_df.columns)
print("Using CoT column:", repr(COT_COL))

# Open-ended answers are free text (not Yes/No), so there's no fixed value set to
# filter against -- just drop rows with a missing/empty answer or a missing/empty CoT.
before = len(open_df)
open_df["answer"]  = open_df["answer"].astype(str).str.strip()
open_df[COT_COL]   = open_df[COT_COL].astype(str).str.strip()
has_answer = (open_df["answer"] != "") & (open_df["answer"].str.lower() != "nan")
has_cot    = (open_df[COT_COL]  != "") & (open_df[COT_COL].str.lower()  != "nan")
print(f"Rows without answer: {int((~has_answer).sum())} | rows without CoT: {int((~has_cot).sum())}")
open_df = open_df[has_answer & has_cot].copy().reset_index(drop=True)

# Sanity check on answer length, since the prompt asks the model for a 1-4 word
# answer: flag (but don't silently drop) anything longer, so you can decide.
word_counts = open_df["answer"].str.split().str.len()
long_mask = word_counts > 4
print(f"Kept {len(open_df)} / {before} rows with a non-empty answer + CoT "
      f"({int(long_mask.sum())} have answers longer than 4 words -- inspect these "
      f"if you want training targets to strictly match the 1-4 word output format).")
if long_mask.any():
    print(open_df.loc[long_mask, "answer"].value_counts().head(10))

# CoT length stats -- the CoT goes into the prompt, so long CoTs increase sequence length / VRAM
cot_words = open_df[COT_COL].str.split().str.len()
print(f"CoT length (words): min={int(cot_words.min())}  mean={cot_words.mean():.0f}  max={int(cot_words.max())}")

IMG_COL = "image_file" if "image_file" in open_df.columns else "img_name"

# Index every image inside the zip by file name (images may be in any sub-folder)
IMG_EXTS = (".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".gif", ".webp")
IMG_INDEX = {os.path.basename(p): p
             for p in glob.glob(os.path.join(EXTRACT_DIR, "**", "*"), recursive=True)
             if p.lower().endswith(IMG_EXTS)}
print(f"Indexed {len(IMG_INDEX)} image files")

def resolve_image_path(split_dir: str, img_name: str) -> str:
    flat_name = os.path.basename(str(img_name))
    candidates = [
        os.path.join(split_dir, str(img_name)),
        os.path.join(split_dir, flat_name),
        os.path.join(split_dir, str(img_name).replace("/", "_")),
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    if flat_name in IMG_INDEX:
        return IMG_INDEX[flat_name]
    raise FileNotFoundError(f"Could not find image for {img_name!r} in {split_dir}")

print("First image:", resolve_image_path(open_df.iloc[0]["split_dir"], open_df.iloc[0][IMG_COL]))


Using zip: /content/Abdomen_Train_with_CoT.zip
Extracted to /content/data
Found 1 CSVs:
  /content/data/Abdomen_Train_with_CoT/CT/train/open_with_CoT.csv
Columns: ['image_file', 'img_id', 'location', 'modality', 'question', 'answer', 'q_lang', 'answer_type', 'content_type', 'base_type', 'qid', 'triple', 'CoT', 'split_dir']
Using CoT column: 'CoT'
Rows without answer: 0 | rows without CoT: 0
Kept 150 / 150 rows with a non-empty answer + CoT (0 have answers longer than 4 words -- inspect these if you want training targets to strictly match the 1-4 word output format).
CoT length (words): min=291  mean=414  max=599
Indexed 92 image files
First image: /content/data/Abdomen_Train_with_CoT/CT/train/xmlab104_source.jpg


In [22]:
# ---------------------------------------------------------------------------
# 4. convert_to_conversation -- instruction = OPEN-question prompt WITH CoT reasoning
#    reference; the assistant target is the ground-truth free-text answer (1-4 words).
# ---------------------------------------------------------------------------
def systemPrompt(question: str, cot: str) -> str:
    return f"""Context:
You are a board-certified radiologist and Medical Visual Question Answering (MedVQA) expert with experience interpreting X-ray, CT, MRI, Ultrasound, and other medical images.

Objective:
Answer the user's question by verifying whether it is supported by the visual evidence in the medical image.

Inputs:
Question: {question}

You have to think step by step.

Instructions:
1. Examine the medical image carefully.
2. Independently determine the relevant visual findings before considering the CoT.
3. Compare your own observations with the provided CoT.
4. If the CoT is inconsistent with the image, disregard it.
5. Answer the question using the following evidence priority:
1. Medical image (highest priority)
2. User question
3. CoT (only if verified by the image)
6. Never fabricate findings or rely on assumptions.

Output Requirements:
- Return ONLY the final answer in 1 to 4 words. Do not explain.

Reasoning Reference:
{cot}

Use it only if it agrees with the image."""

def convert_to_conversation(sample):
    instruction = systemPrompt(sample["question"], sample[COT_COL])
    image_path = resolve_image_path(sample["split_dir"], sample[IMG_COL])
    image = Image.open(image_path).convert("RGB")

    conversation = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": instruction},
                {"type": "image", "image": image},
            ],
        },
        {
            "role": "assistant",
            "content": [{"type": "text", "text": sample["answer"]}],
        },
    ]
    return {"messages": conversation}
pass

print("Converting rows to Unsloth chat format...")
converted_dataset = [convert_to_conversation(row) for _, row in open_df.iterrows()]
print(f"Done. {len(converted_dataset)} examples ready.")
print("Example:", converted_dataset[0]["messages"])


Converting rows to Unsloth chat format...
Done. 150 examples ready.
Example: [{'role': 'user', 'content': [{'type': 'text', 'text': "Context:\nYou are a board-certified radiologist and Medical Visual Question Answering (MedVQA) expert with experience interpreting X-ray, CT, MRI, Ultrasound, and other medical images.\n\nObjective:\nAnswer the user's question by verifying whether it is supported by the visual evidence in the medical image.\n\nInputs:\nQuestion: What modality is used to take this image?\n\nYou have to think step by step.\n\nInstructions:\n1. Examine the medical image carefully.\n2. Independently determine the relevant visual findings before considering the CoT.\n3. Compare your own observations with the provided CoT.\n4. If the CoT is inconsistent with the image, disregard it.\n5. Answer the question using the following evidence priority:\n1. Medical image (highest priority)\n2. User question\n3. CoT (only if verified by the image)\n6. Never fabricate findings or rely on 

## Per-model steps (each cell = one step of the original notebook)


In [23]:
# ---------------------------------------------------------------------------
# 1. Load model
# ---------------------------------------------------------------------------
def load_model(repo_id):
    extra = {}
    if "Qwen3-VL" in repo_id:
        extra["device_map"] = {"": 0}   # Qwen3-VL vision encoder breaks when split across GPUs
    model, tokenizer = FastVisionModel.from_pretrained(
        repo_id,
        load_in_4bit = True, # Use 4bit to reduce memory use. False for 16bit LoRA.
        use_gradient_checkpointing = "unsloth", # True or "unsloth" for long context
        token = HF_TOKEN,
        **extra,
    )
    return model, tokenizer


In [24]:
# ---------------------------------------------------------------------------
# 2. Attach LoRA adapters
# ---------------------------------------------------------------------------
def attach_lora(model, model_name):
    model = FastVisionModel.get_peft_model(
        model,
        finetune_vision_layers=(model_name != "MedGemma-4B"),   # T4 has no bf16: keep MedGemma's vision tower frozen
        finetune_language_layers=True,
        finetune_attention_modules=True,
        finetune_mlp_modules=True,
        r=16,
        lora_alpha=16,
        lora_dropout=0,
        bias="none",
        random_state=3407,
        use_rslora=False,
        loftq_config=None,
    )
    if model_name == "MedGemma-4B":
        # T4 trains Gemma3 in float32; upcast any bf16 weights left in the vision tower so layer_norm dtypes match
        for p in model.parameters():
            if p.dtype == torch.bfloat16:
                p.data = p.data.float()
        for b in model.buffers():
            if b.dtype == torch.bfloat16:
                b.data = b.data.float()
    return model


In [25]:
# ---------------------------------------------------------------------------
# 5. Quick check BEFORE fine-tuning (same as the Unsloth notebook does --
#    run one inference with the base model to see what it currently outputs)
# ---------------------------------------------------------------------------
def check_before(model, tokenizer):
    FastVisionModel.for_inference(model)
    sample = open_df.iloc[0]
    test_instruction = systemPrompt(sample["question"], sample[COT_COL])
    test_image = Image.open(resolve_image_path(sample["split_dir"], sample[IMG_COL])).convert("RGB")

    messages = [{"role": "user", "content": [
        {"type": "image"}, {"type": "text", "text": test_instruction}
    ]}]
    input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
    inputs = tokenizer(test_image, input_text, add_special_tokens=False, return_tensors="pt").to("cuda")
    output_ids = model.generate(**inputs, max_new_tokens=32, use_cache=True, temperature=1.5, min_p=0.1)
    print("\nBEFORE fine-tuning, model output:")
    print(tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))
    print("Ground truth:", sample["answer"])
    return inputs, sample


In [26]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bf16_supported
from unsloth.trainer import UnslothVisionDataCollator

def build_trainer(model, tokenizer, model_name):
    FastVisionModel.for_training(model)

    # CoT prompt is ~2x longer; with UNSLOTH_RETURN_LOGITS=1 the full fp32 logits are
    # materialised for the loss, so batch 2 OOMs on a T4 for the 7B/8B/11B backbones.
    # Batch 1 x 8 accumulation keeps the same effective batch (8) and the same step count.
    if model_name == "MedGemma-4B":
        bs, ga = 2, 4
    else:
        bs, ga = 1, 8

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        data_collator=UnslothVisionDataCollator(model, tokenizer),
        train_dataset=converted_dataset,
        args=SFTConfig(
            per_device_train_batch_size=bs,
            gradient_accumulation_steps=ga,
            warmup_steps=5,
            #max_steps=30,                  # quick test run -- comment out for full training
            num_train_epochs=1,
            learning_rate=2e-4,
            fp16=not is_bf16_supported(),
            bf16=is_bf16_supported(),
            logging_steps=1,
            optim="adamw_8bit",
            weight_decay=0.01,
            lr_scheduler_type="linear",
            seed=3407,
            output_dir="outputs",
            report_to="none",

            remove_unused_columns=False,
            dataset_text_field="",
            dataset_kwargs={"skip_prepare_dataset": True},
            dataset_num_proc=4,
            max_length=4096,
        ),
    )
    return trainer

In [27]:
# ---------------------------------------------------------------------------
# 7. Check AFTER fine-tuning -- same sample as before, compare outputs
# ---------------------------------------------------------------------------
def check_after(model, tokenizer, inputs, sample):
    FastVisionModel.for_inference(model)
    output_ids = model.generate(**inputs, max_new_tokens=32, use_cache=True, temperature=0.3, min_p=0.1)
    print("\nAFTER fine-tuning, model output:")
    print(tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))
    print("\n(Ground truth answer it was trained on:)")
    print(sample["answer"])


In [28]:
# ---------------------------------------------------------------------------
# 8. Save the LoRA adapter locally
# ---------------------------------------------------------------------------
def save_local(model, tokenizer, local_dir):
    model.save_pretrained(local_dir)
    tokenizer.save_pretrained(local_dir)
    print(f"\nSaved locally to ./{local_dir}")


In [29]:
# ---------------------------------------------------------------------------
# 9. Push to Hugging Face Hub
# ---------------------------------------------------------------------------
def push_to_hf(model, tokenizer, hub_repo):
    model.push_to_hub(hub_repo, token=HF_TOKEN)
    tokenizer.push_to_hub(hub_repo, token=HF_TOKEN)
    print(f"Pushed -> https://huggingface.co/{hub_repo}")


In [30]:
import sys

def free_gpu():
    for v in ["trainer", "trainer_stats", "model", "tokenizer", "inputs", "sample"]:
        if v in globals(): del globals()[v]
    # IPython keeps the last exception's traceback (and every tensor its frames reference) alive
    sys.last_type = sys.last_value = sys.last_traceback = None
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.ipc_collect()
    shutil.rmtree(os.path.expanduser("~/.cache/huggingface/hub"), ignore_errors=True)
    print("GPU free after cleanup:", [f"{torch.cuda.mem_get_info(i)[0]/1e9:.1f} GB" for i in range(torch.cuda.device_count())])

## Run all models (one after another; each pushed to the Hub before the next starts)


In [31]:
from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)
results = {}

for model_name, repo_id in REPO_IDS.items():
    HUB_REPO  = f"{HF_USERNAME}/{DATASET_TAG}_open_CoT_{model_name}_lora"
    LOCAL_DIR = f"{DATASET_TAG}_open_CoT_{model_name}_lora"

    if api.repo_exists(HUB_REPO):
        print(f"\n[SKIP] {model_name} already on Hub: https://huggingface.co/{HUB_REPO}")
        results[model_name] = "skipped"
        continue

    # ~2 GB leaks per model on a T4 even after cleanup. Stop early instead of burning
    # 15 min on a run that will OOM; finished models are skipped on re-run.
    MIN_FREE_GB = 13.0
    free_gb = torch.cuda.mem_get_info(0)[0] / 1e9
    if free_gb < MIN_FREE_GB:
        print(f"\n[STOP] Only {free_gb:.1f} GB free (need ~{MIN_FREE_GB:.0f} GB for {model_name}).")
        print("       Runtime -> Restart session, then run this cell again.")
        results[model_name] = f"not run (only {free_gb:.1f} GB free -- restart session)"
        break

    print("\n" + "="*70)
    print(f"MODEL: {model_name}  ({repo_id})  ->  {HUB_REPO}")
    print("="*70)

    try:
        model, tokenizer = load_model(repo_id)                          # 1
        model            = attach_lora(model, model_name)               # 2
        inputs, sample   = check_before(model, tokenizer)               # 5
        trainer          = build_trainer(model, tokenizer, model_name)  # 6
        trainer_stats    = trainer.train()
        check_after(model, tokenizer, inputs, sample)                   # 7
        save_local(model, tokenizer, LOCAL_DIR)                         # 8
        push_to_hf(model, tokenizer, HUB_REPO)                          # 9
        results[model_name] = "OK"
    except Exception as e:
        import traceback; traceback.print_exc()
        results[model_name] = f"FAILED: {e}"

    free_gpu()                                                          # 10

print("\n================ SUMMARY ================")
for k, v in results.items():
    print(f"{k:15s} {v}")


[SKIP] MedGemma-4B already on Hub: https://huggingface.co/Arup330/Abdomen_open_CoT_MedGemma-4B_lora

[SKIP] Qwen2.5-VL-7B already on Hub: https://huggingface.co/Arup330/Abdomen_open_CoT_Qwen2.5-VL-7B_lora

[STOP] Only 7.1 GB free (need ~13 GB for Qwen3-VL-8B).
       Runtime -> Restart session, then run this cell again.

================ SUMMARY ================
MedGemma-4B     skipped
Qwen2.5-VL-7B   skipped
Qwen3-VL-8B     not run (only 7.1 GB free -- restart session)
